# Comparación con pseudoausencias cercanas

**Objetivo:** RF y MLP con atención; 8 variables, pseudoausencias cercanas.

Este cuaderno contiene el código completo del experimento `train_realistic_models.py`,
dividido en etapas y acompañado de explicaciones. No necesita ZIP, clonar un
repositorio ni ejecutar un script oculto. Ejecuta las celdas en orden.

**Recorrido:** entorno → carga e inspección → preparación → modelo o análisis →
evaluación → gráficos → descarga y colaboración.

**Entradas:** `dataset_entrenamiento_sierra_realista.csv`.

Los datos no se incluyen. Se conservan los valores predeterminados del protocolo histórico.
Las salidas se generarán al ejecutar; las figuras históricas no están incrustadas.


**Interpretación:** los pesos de atención, la importancia Gini y SHAP son indicadores distintos. No equivalen a causas físicas. El módulo de atención se conserva para documentar el experimento; su revisión debe evaluarse como una variante nueva.

## 1. Entorno y parámetros
Modifica los parámetros aquí para que una colaboración sea fácil de revisar. Los valores originales siguen seleccionados. Cada ejecución crea una carpeta nueva.

In [ ]:
# Instala únicamente las bibliotecas que falten en el entorno.
import importlib.util
import importlib.metadata
import subprocess
import sys
import os
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display, Image

DEPENDENCIAS = {'numpy': 'numpy', 'pandas': 'pandas', 'sklearn': 'scikit-learn', 'matplotlib': 'matplotlib', 'torch': 'torch'}
faltantes = [paquete for modulo, paquete in DEPENDENCIAS.items()
             if importlib.util.find_spec(modulo) is None]
if faltantes:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *faltantes], check=True)

CONFIG = dict(seed=42, rf_trees=100, rf_depth=15, epochs=50,
              learning_rate=0.002, batch_size=256)
EXPERIMENTO = 'hard_negatives'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
DATA_DIR = Path(globals().get('DATA_DIR_OVERRIDE', 'data')).resolve()
OUTPUT_DIR = Path(globals().get('OUTPUT_DIR_OVERRIDE', Path('results') / EXPERIMENTO / RUN_ID)).resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLBACKEND', 'Agg')
print('Parámetros:', CONFIG)
print('Salidas:', OUTPUT_DIR)

def mostrar_figura(ruta):
    # Conserva la figura en disco y también la muestra en el cuaderno.
    if str(ruta).lower().endswith('.png'):
        display(Image(filename=str(ruta)))


## 2. Carga e inspección de los datos
En Colab, la siguiente celda ofrece el selector si faltan archivos. Carga solo los CSV indicados, conservando sus nombres. En un entorno local, colócalos en `data/`. Para el dataset realista usa la copia de la subcarpeta `resultados` original.

In [ ]:
import pandas as pd

ARCHIVOS_REQUERIDOS = ['dataset_entrenamiento_sierra_realista.csv']
def localizar(nombre):
    candidatos = [DATA_DIR / nombre, DATA_DIR / 'resultados' / nombre]
    return next((p for p in candidatos if p.is_file()), candidatos[0])

pendientes = [n for n in ARCHIVOS_REQUERIDOS if not localizar(n).is_file()]
if pendientes:
    print('Archivos requeridos:', pendientes)
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError('Coloca los CSV en ' + str(DATA_DIR))
    cargados = files.upload()
    for nombre, contenido in cargados.items():
        if Path(nombre).name != nombre or nombre not in ARCHIVOS_REQUERIDOS:
            raise ValueError('Archivo inesperado: ' + nombre)
        (DATA_DIR / nombre).write_bytes(contenido)
    del cargados

DATA_FILES = {n: localizar(n) for n in ARCHIVOS_REQUERIDOS}
if any(not p.is_file() for p in DATA_FILES.values()):
    raise FileNotFoundError('Faltan CSV; repite esta celda y carga los indicados.')

# Inspección visible: dimensiones, tipos, valores faltantes y balance de clases.
for nombre, ruta in DATA_FILES.items():
    tabla = pd.read_csv(ruta)
    print('\n', nombre, '| filas y columnas:', tabla.shape)
    display(tabla.head())
    display(pd.DataFrame({'tipo': tabla.dtypes.astype(str), 'nulos': tabla.isna().sum()}))
    if 'clase' in tabla.columns:
        display(tabla['clase'].value_counts(dropna=False).rename('filas').to_frame())

REGISTRO = dict(experimento=EXPERIMENTO, parametros=CONFIG, run_id=RUN_ID,
    input_sha256={n: hashlib.sha256(p.read_bytes()).hexdigest() for n, p in DATA_FILES.items()},
    versiones={p: importlib.metadata.version(p) for p in DEPENDENCIAS.values()},
    estado='iniciado')
(OUTPUT_DIR / 'ejecucion.json').write_text(json.dumps(REGISTRO, ensure_ascii=False, indent=2), encoding='utf-8')


In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
BASE_DIR = str(OUTPUT_DIR)
FILE_REALISTA = str(DATA_FILES['dataset_entrenamiento_sierra_realista.csv'])
FILE_CSV_FACTORES = str(OUTPUT_DIR / 'comparativa_factores_realista.csv')
FILE_PNG_FACTORES = str(OUTPUT_DIR / 'comparativa_modelos_factores_realista.png')
FILE_PNG_VALIDACION = str(OUTPUT_DIR / 'comparativa_validacion_espacial_realista.png')
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])


### Definición: `FeatureSelfAttention`
Esta definición queda visible para revisar su arquitectura y proponer variantes. Sus valores originales se conservan.

In [ ]:
class FeatureSelfAttention(nn.Module):
    def __init__(self, input_dim):
        super(FeatureSelfAttention, self).__init__()
        self.query = nn.Linear(input_dim, input_dim, bias=False)
        self.key = nn.Linear(input_dim, input_dim, bias=False)
        self.value = nn.Linear(input_dim, input_dim, bias=False)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)
        attention_scores = torch.matmul(q.unsqueeze(2), k.unsqueeze(1)) / np.sqrt(x.shape[1])
        attention_weights = self.softmax(attention_scores.mean(dim=2))
        attended_x = v * attention_weights
        return attended_x, attention_weights


### Definición: `FireAttentionMLP`
Esta definición queda visible para revisar su arquitectura y proponer variantes. Sus valores originales se conservan.

In [ ]:
class FireAttentionMLP(nn.Module):
    def __init__(self, input_dim):
        super(FireAttentionMLP, self).__init__()
        self.attention = FeatureSelfAttention(input_dim)
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        attended_x, weights = self.attention(x)
        output = self.classifier(attended_x)
        return output, weights


## 4. Desarrollo del experimento
Las siguientes celdas contienen el cuerpo del análisis original, con comentarios y código editable. Puedes inspeccionar las variables entre pasos. Ejecuta nuevamente desde la carga de datos si cambias una transformación previa.

### Etapa 1: Carga y preparación de datos
Comprueba las columnas, las filas descartadas y el significado de la etiqueta antes de entrenar. El código conserva la selección de variables del experimento original.

In [ ]:
print("="*75)
print("  ENTRENAMIENTO REALISTA CON AUSENCIAS CERCANAS (HARD NEGATIVE BUFFER)")
print("="*75)

if not os.path.exists(FILE_REALISTA):
    print(f"[ERROR] No se encuentra el archivo: {FILE_REALISTA}")
    raise ValueError("Revisa el archivo y las columnas indicadas en la salida anterior.")
df = pd.read_csv(FILE_REALISTA)
features = ['elevacion', 'pendiente', 'orientacion', 'ndvi', 'evi', 'nbr', 'temp_c', 'precip_mm']

nombres_es = {
    'temp_c': 'Temperatura',
    'precip_mm': 'Precipitación',
    'elevacion': 'Elevación',
    'ndvi': 'NDVI (Verdor)',
    'nbr': 'NBR (Humedad)',
    'evi': 'EVI (Biomasa)',
    'pendiente': 'Pendiente',
    'orientacion': 'Orientación'
}

# Limpiar nulos
df = df.dropna(subset=features)
X = df[features].values
y = df['clase'].values

print(f"[INFO] Dataset cargado: {len(df)} muestras.")


### Etapa 2: Ajuste del modelo
Aquí se aprende a partir de las muestras de entrenamiento. Revisa los hiperparámetros visibles y evita ajustar decisiones usando repetidamente los resultados del conjunto de prueba.

In [ ]:
print(f"       - Presencias (Incendios): {sum(y == 1)}")
print(f"       - Ausencias Difíciles:    {sum(y == 0)}")

# 1. DIVISIÓN ALEATORIA ESTRATIFICADA (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=CONFIG['seed'], stratify=y)

# Escalado
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# --- MODELO 1: RANDOM FOREST ---
print("\n[1/2] Entrenando Random Forest Classifier...")
rf = RandomForestClassifier(n_estimators=CONFIG['rf_trees'], max_depth=CONFIG['rf_depth'], random_state=CONFIG['seed'], n_jobs=-1)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]

rf_acc = accuracy_score(y_test, rf_pred)
rf_prec = precision_score(y_test, rf_pred)
rf_rec = recall_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_proba)
rf_cm = confusion_matrix(y_test, rf_pred)

# --- MODELO 2: RED DE AUTOTENSIÓN (PYTORCH) ---
print("[2/2] Entrenando Red Neuronal de Autotensión (PyTorch)...")


### Etapa 3: Entrenamiento de la red
Se optimizan los pesos con los datos de entrenamiento. El registro muestra la pérdida de la última tanda; no es una métrica de validación. Mantén fijo el conjunto de prueba al comparar configuraciones.

In [ ]:

train_dataset = TensorDataset(torch.FloatTensor(X_train_s), torch.FloatTensor(y_train).unsqueeze(1))
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
X_test_t = torch.FloatTensor(X_test_s)

model = FireAttentionMLP(input_dim=len(features))
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=1e-5)

model.train()
for epoca in range(CONFIG['epochs']):
    for bx, by in train_loader:
        optimizer.zero_grad()
        out, _ = model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
    if epoca == 0 or (epoca + 1) % 10 == 0 or epoca + 1 == CONFIG['epochs']:
        print(f"Época {epoca + 1}/{CONFIG['epochs']} | pérdida última tanda: {float(loss.detach()):.5f}")

model.eval()
with torch.no_grad():
    nn_proba_t, nn_weights_t = model(X_test_t)
    nn_proba = nn_proba_t.numpy().flatten()
    nn_pred = (nn_proba >= 0.5).astype(int)

    nn_attn_mean = nn_weights_t.mean(dim=0).numpy()
    nn_importances = nn_attn_mean / np.sum(nn_attn_mean)


### Etapa 4: Evaluación
Estas métricas pertenecen al protocolo y a la partición de este cuaderno. Revisa precisión, recall y matriz de confusión además del accuracy.

In [ ]:

nn_acc = accuracy_score(y_test, nn_pred)
nn_prec = precision_score(y_test, nn_pred)
nn_rec = recall_score(y_test, nn_pred)
nn_f1 = f1_score(y_test, nn_pred)
nn_auc = roc_auc_score(y_test, nn_proba)
nn_cm = confusion_matrix(y_test, nn_pred)

# IMPRIMIR RESULTADOS REALISTAS
print("\n" + "="*65)
print("  RESULTADOS REALISTAS (CON AUSENCIAS DIFICILES POR BUFFER 1-8KM)")
print("="*65)
print(f"Random Forest  -> Accuracy: {rf_acc:.4f} ({rf_acc*100:.2f}%) | ROC-AUC: {rf_auc:.4f}")
print(f"Attention Net  -> Accuracy: {nn_acc:.4f} ({nn_acc*100:.2f}%) | ROC-AUC: {nn_auc:.4f}")

print("\n[Matriz de Confusión Random Forest]:")
print(f"  TN: {rf_cm[0][0]}, FP: {rf_cm[0][1]} | FN: {rf_cm[1][0]}, TP: {rf_cm[1][1]}")

print("\n[Matriz de Confusión Red de Atención]:")
print(f"  TN: {nn_cm[0][0]}, FP: {nn_cm[0][1]} | FN: {nn_cm[1][0]}, TP: {nn_cm[1][1]}")

# IMPORTANCIA DE VARIABLES REALISTA
rf_importances = rf.feature_importances_
df_comp = pd.DataFrame({
    'Variable': [nombres_es[f] for f in features],
    'Random Forest (Gini)': rf_importances * 100,
    'Attention Net (Pesos)': nn_importances * 100
}).sort_values(by='Attention Net (Pesos)', ascending=False).reset_index(drop=True)


### Etapa 5: Representación y exportación
Lee los ejes y el conjunto de datos utilizado. Las figuras PNG se muestran al guardarse; al final también puedes inspeccionar todas las salidas y descargarlas.

In [ ]:

print("\n" + "="*65)
print("  IMPORTANCIA COMPARATIVA DE LAS 8 VARIABLES (%)")
print("="*65)
print(df_comp.round(2).to_string(index=False))
df_comp.to_csv(FILE_CSV_FACTORES, index=False)

# GRAFICAR IMPORTANCIA DE VARIABLES REALISTA
plt.figure(figsize=(12, 7), dpi=300)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

y_pos = np.arange(len(features))
width = 0.35
df_plot = df_comp.sort_values(by='Attention Net (Pesos)', ascending=True).reset_index(drop=True)

plt.barh(y_pos - width/2, df_plot['Random Forest (Gini)'], width, label='Random Forest (Gini)', color='#d95f0e')
plt.barh(y_pos + width/2, df_plot['Attention Net (Pesos)'], width, label='Red de Atención (Pesos)', color='#2b8cbe')

plt.yticks(y_pos, df_plot['Variable'], fontsize=11, fontweight='semibold')
plt.xlabel('Importancia Relativa / Peso del Factor (%)', fontsize=12, fontweight='bold', labelpad=10)
plt.title('Factores de Incendios en la Sierra Ecuatoriana (Dataset Realista)', fontsize=14, fontweight='bold', pad=15)
plt.legend(loc='lower right', fontsize=11)

ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)


### Etapa 6: Representación y exportación
Lee los ejes y el conjunto de datos utilizado. Las figuras PNG se muestran al guardarse; al final también puedes inspeccionar todas las salidas y descargarlas.

In [ ]:
ax.spines['left'].set_visible(False)
ax.xaxis.grid(True, linestyle='--', alpha=0.5, color='#cccccc')

plt.tight_layout()
plt.savefig(FILE_PNG_FACTORES, bbox_inches='tight')
mostrar_figura(FILE_PNG_FACTORES)
plt.close()

print(f"\n[OK] Gráfico comparativo guardado en: {FILE_PNG_FACTORES}")
print("="*75)


## 5. Inspeccionar y guardar resultados
Las tablas se muestran debajo. El registro identifica parámetros, versiones y hashes. Esta celda marca el registro como completado: ejecútala solo cuando hayan terminado todas las etapas anteriores.

In [ ]:
for ruta in sorted(OUTPUT_DIR.glob('*.csv')):
    print(ruta.name)
    display(pd.read_csv(ruta).head(20))
REGISTRO['estado'] = 'completado'
(OUTPUT_DIR / 'ejecucion.json').write_text(json.dumps(REGISTRO, ensure_ascii=False, indent=2), encoding='utf-8')
print('Salidas:', [p.name for p in sorted(OUTPUT_DIR.iterdir())])


In [ ]:
# Descarga las salidas antes de cerrar una sesión de Colab.
import shutil
paquete = shutil.make_archive(str(OUTPUT_DIR.parent / (EXPERIMENTO + '-' + RUN_ID)), 'zip', OUTPUT_DIR)
try:
    from google.colab import files
except ImportError:
    print('ZIP guardado en:', paquete)
else:
    files.download(paquete)


## 6. Proponer una colaboración

1. Plantea una pregunta concreta: por ejemplo, el efecto de un hiperparámetro o de una variable.
2. Conserva una ejecución con la configuración original y luego cambia una sola decisión.
3. Registra el hash del dataset, la partición, la semilla y todas las métricas relevantes.
4. En GitHub, abre un Issue para discutir la propuesta. Trabaja en un fork o en una rama
   con permisos y envía un pull request que explique el cambio y su evidencia.
5. Antes del commit, borra las salidas del cuaderno y revisa que no incluyan datos privados.
   Adjunta resultados únicamente cuando tengas permiso para compartirlos.

Edita la siguiente celda Markdown para documentar tu propuesta. No se envía nada automáticamente.

### Registro de la propuesta

- **Autor y objetivo:**
- **Hipótesis:**
- **Parámetro o código modificado:**
- **Dataset y hash:**
- **Semilla y partición:**
- **Resultado original y resultado de la variante:**
- **Limitaciones y conclusión:**
